In [1]:
!pip install boto3 psycopg2-binary pyarrow

In [3]:
import pandas as pd
import psycopg2
import boto3
from io import BytesIO

conn = psycopg2.connect(
    host="postgres",
    port=5432,
    database="oil_gas",
    user="oil",
    password="oil123"
)

s3 = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin'
)

bucket = 'oil-data'
try:
    s3.create_bucket(Bucket=bucket)
except:
    pass

tables = ['wells', 'production', 'well_telemetry', 'well_targets', 'pump_sensors', 'pump_failures', 'deliveries']

for table in tables:
    df = pd.read_sql(f"SELECT * FROM {table}", conn)
    
    date_col = None
    if 'date' in df.columns:
        date_col = 'date'
    elif 'production_date' in df.columns:
        date_col = 'production_date'
    elif 'timestamp' in df.columns:
        date_col = 'timestamp'
    
    if date_col:
        df['year'] = pd.to_datetime(df[date_col]).dt.year
        df['month'] = pd.to_datetime(df[date_col]).dt.month
        for (year, month), group in df.groupby(['year', 'month']):
            buf = BytesIO()
            group.to_parquet(buf, index=False)
            buf.seek(0)
            key = f"raw/{table}/year={year}/month={month}/data.parquet"
            s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())
    else:
        buf = BytesIO()
        df.to_parquet(buf, index=False)
        buf.seek(0)
        key = f"raw/{table}/data.parquet"
        s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())

conn.close()

/tmp/ipykernel_7625/954892978.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {table}", conn)
/tmp/ipykernel_7625/954892978.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {table}", conn)
/tmp/ipykernel_7625/954892978.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {table}", conn)
/tmp/ipykernel_7625/954892978.py:30: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string 